# 15 — Rolling Hash (Deep Dive)

## Learning goals
- Understand polynomial hashing and modular arithmetic.
- Build a hash for a string and update it as a window moves.
- Implement Rabin–Karp substring search.
- Understand collisions and verify hash matches.
- Recognize rolling hash + binary search patterns in advanced problems.

**Important:** Rolling hash is a useful string algorithm, but a hash match is not proof of equality. Verify candidates when correctness requires it.

## 1. Why do we need rolling hash?

Suppose we want to compare every substring of length `m` in a text of length `n` with a pattern.

If we rebuild each substring's hash from scratch, each window can cost `O(m)`, giving `O(nm)` work.

Rolling hash reuses the previous window's hash. After the first window, each shift can be processed in `O(1)` arithmetic operations.

## 2. Polynomial hash

Map each character to an integer and treat the string like a number in base `B`.

For characters with numeric values `c0, c1, ..., c(m-1)`:

\[
H = (c_0 B^{m-1} + c_1 B^{m-2} + \cdots + c_{m-1}) \bmod M
\]

- `B`: base.
- `M`: modulus, usually a large prime.
- `c`: character encoding.

The modulus keeps the number bounded. Different strings may still produce the same hash.

In [1]:
def polynomial_hash(s: str, base: int = 31, mod: int = 1_000_000_007) -> int:
    value = 0

    for ch in s:
        value = (value * base + ord(ch)) % mod

    return value

print(polynomial_hash("abc"))
print(polynomial_hash("abd"))

96354
96355


## 3. Derive the rolling update

For a window of length `m`:

`H = c0 * B^(m-1) + c1 * B^(m-2) + ... + c(m-1)`

When the window moves one step:
1. Subtract the outgoing character multiplied by `B^(m-1)`.
2. Multiply the remaining value by `B`.
3. Add the incoming character.
4. Take modulo `M`.

Formula:

\[
H_{next} = ((H - outgoing \cdot B^{m-1}) \cdot B + incoming) \bmod M
\]

In Python, `% mod` returns a non-negative remainder for a positive modulus, so it safely normalizes after subtraction.

In [2]:
def rolling_hashes(s: str, m: int, base: int = 31,
                   mod: int = 1_000_000_007) -> list[int]:
    n = len(s)

    if not 1 <= m <= n:
        return []

    highest_power = pow(base, m - 1, mod)

    current = 0
    for ch in s[:m]:
        current = (current * base + ord(ch)) % mod

    result = [current]

    for right in range(m, n):
        outgoing = ord(s[right - m])
        incoming = ord(s[right])

        current = (current - outgoing * highest_power) % mod
        current = (current * base + incoming) % mod

        result.append(current)

    return result

print(rolling_hashes("abcdef", 3))

[96354, 97347, 98340, 99333]


## 4. Verify rolling hashes against fresh hashes

A good debugging technique is to compare every rolling hash with a hash calculated directly from that window.

This helps catch mistakes in the outgoing-character power or update order.

In [3]:
text = "abracadabra"
m = 4

actual = rolling_hashes(text, m)
expected = [
    polynomial_hash(text[i:i + m])
    for i in range(len(text) - m + 1)
]

print(actual == expected)  # True

True


## 5. Rabin–Karp substring search

Goal: return every starting index where `pattern` occurs in `text`.

Steps:
1. Hash the pattern.
2. Hash the first text window of the same length.
3. Compare the hashes.
4. If hashes match, compare the actual substring to avoid a collision-based false match.
5. Roll the text-window hash forward.

The explicit substring comparison guarantees correctness even if a collision occurs.

In [4]:
def rabin_karp(text: str, pattern: str) -> list[int]:
    n, m = len(text), len(pattern)

    if m == 0:
        return list(range(n + 1))
    if m > n:
        return []

    base = 256
    mod = 1_000_000_007
    highest_power = pow(base, m - 1, mod)

    pattern_hash = 0
    window_hash = 0

    for i in range(m):
        pattern_hash = (pattern_hash * base + ord(pattern[i])) % mod
        window_hash = (window_hash * base + ord(text[i])) % mod

    answer = []

    for start in range(n - m + 1):
        if window_hash == pattern_hash:
            if text[start:start + m] == pattern:
                answer.append(start)

        if start < n - m:
            outgoing = ord(text[start])
            incoming = ord(text[start + m])

            window_hash = (window_hash - outgoing * highest_power) % mod
            window_hash = (window_hash * base + incoming) % mod

    return answer

print(rabin_karp("ababcabcabababd", "ababd"))  # [10]
print(rabin_karp("aaaaa", "aa"))               # [0, 1, 2, 3]
print(rabin_karp("hello", "xyz"))              # []

[10]
[0, 1, 2, 3]
[]


## 6. Why collision verification matters

A collision means two different strings have the same hash.

Therefore:
- Different hashes prove the strings are different (for the same hash scheme).
- Equal hashes only mean the strings are possible matches.
- Compare the actual substring after a hash match.

Using a larger modulus or two independent hashes makes accidental collisions less likely, but direct verification is the strongest correctness check.

## 7. Prefix hashes — answer substring hash queries

A prefix hash lets us calculate a substring hash using two prefix values.

Define:

`prefix[i] = hash of the first i characters`

With polynomial powers increasing from left to right:

\[
prefix[r] - prefix[l] \cdot B^{r-l}
\]

is the hash of substring `s[l:r]`, modulo `M`.

This representation is useful for many substring comparisons. The exact formula depends on the chosen hash convention.

In [5]:
class PrefixRollingHash:
    def __init__(self, s: str, base: int = 31, mod: int = 1_000_000_007):
        self.base = base
        self.mod = mod
        n = len(s)

        self.prefix = [0] * (n + 1)
        self.power = [1] * (n + 1)

        for i, ch in enumerate(s):
            self.prefix[i + 1] = (
                self.prefix[i] * base + ord(ch)
            ) % mod
            self.power[i + 1] = self.power[i] * base % mod

    def substring_hash(self, left: int, right: int) -> int:
        # Hash of s[left:right], where right is exclusive.
        if not (0 <= left <= right < len(self.prefix)):
            raise IndexError("Invalid substring bounds")

        value = (
            self.prefix[right]
            - self.prefix[left] * self.power[right - left]
        ) % self.mod

        return value

s = "abracadabra"
hasher = PrefixRollingHash(s)

print(hasher.substring_hash(0, 3))
print(hasher.substring_hash(7, 10))
print(hasher.substring_hash(0, 3) == hasher.substring_hash(7, 10))  # True

96369
96369
True


## 8. Longest duplicate substring — binary search + rolling hash

Problem idea: find the longest substring that appears at least twice.

A key observation:
- If a duplicate substring of length `L` exists, a duplicate substring of every smaller positive length also exists.
- If no duplicate substring of length `L` exists, no longer length can have one.

This monotonic property allows binary search on the length.

For each candidate length:
1. Compute the hash of the first window.
2. Roll through all windows of that length.
3. Track hashes already seen.
4. On a repeated hash, verify the actual substring to protect correctness.

For a production-grade implementation, consider double hashing or more advanced suffix-array/suffix-automaton approaches.

In [6]:
def longest_duplicate_substring(s: str) -> str:
    n = len(s)
    if n < 2:
        return ""

    base = 256
    mod = 1_000_000_007

    def find_duplicate(length: int):
        if length == 0:
            return 0

        highest_power = pow(base, length - 1, mod)
        current = 0

        for ch in s[:length]:
            current = (current * base + ord(ch)) % mod

        seen = {current: [0]}

        for right in range(length, n):
            outgoing = ord(s[right - length])
            incoming = ord(s[right])

            current = (current - outgoing * highest_power) % mod
            current = (current * base + incoming) % mod

            start = right - length + 1

            if current in seen:
                candidate = s[start:start + length]
                for previous_start in seen[current]:
                    if s[previous_start:previous_start + length] == candidate:
                        return start
                seen[current].append(start)
            else:
                seen[current] = [start]

        return None

    low, high = 1, n - 1
    best_start = 0
    best_length = 0

    while low <= high:
        mid = (low + high) // 2
        found = find_duplicate(mid)

        if found is not None:
            best_start = found
            best_length = mid
            low = mid + 1
        else:
            high = mid - 1

    return s[best_start:best_start + best_length]

print(longest_duplicate_substring("banana"))  # "ana"
print(longest_duplicate_substring("abcd"))    # 

ana



## 9. Complexity analysis

Let `n` be text length and `m` be pattern/window length.

| Operation | Time |
|---|---:|
| Build one polynomial hash | `O(m)` |
| Build first window hash | `O(m)` |
| Update one rolling window | `O(1)` |
| Scan all windows | `O(n)` hash operations |
| Rabin–Karp with verification | Expected `O(n + m)` with few collisions; worst case can be `O(nm)` |
| Build prefix hashes and powers | `O(n)` |
| Query one substring hash | `O(1)` |

The binary-search + rolling-hash duplicate-substring approach is commonly expected around `O(n log n)` hash work, but collision verification can increase worst-case time.

## 10. Double hashing

To reduce accidental collisions, compute two hashes using different moduli (and often different bases).

A pair `(hash1, hash2)` is a stronger signature than one hash. Still, if correctness must be guaranteed, verify the original substring when both hashes match.

Do not confuse “very unlikely collision” with “impossible collision”.

## 11. Common mistakes

1. Using the wrong power for the outgoing character.
2. Removing the outgoing character after multiplying, in the wrong order.
3. Forgetting modulo after subtraction.
4. Comparing only hashes and never verifying matches.
5. Using inconsistent character encodings or bases between pattern and text.
6. Mishandling empty patterns or patterns longer than the text.
7. Assuming a single hash is collision-free.
8. Mixing inclusive and exclusive substring boundaries.

## 12. When should you use rolling hash?

Good fit:
- Repeated fixed-length substring comparisons.
- Rabin–Karp pattern matching.
- Duplicate substring detection.
- Binary search over substring length when the “exists duplicate of length L” condition is monotonic.

Other algorithms may be better:
- KMP for one-pattern exact matching with deterministic linear time.
- Z algorithm for prefix-match information.
- Trie for prefix-based dictionary lookup.
- Suffix array/tree/automaton for extensive substring queries.

Choose based on the required operation, constraints, and collision requirements.

## 13. Practice questions

1. [28. Find the Index of the First Occurrence in a String](https://leetcode.com/problems/find-the-index-of-the-first-occurrence-in-a-string/) — compare KMP and Rabin–Karp.
2. [1044. Longest Duplicate Substring](https://leetcode.com/problems/longest-duplicate-substring/) — binary search + rolling hash.
3. [1062. Longest Repeating Substring](https://leetcode.com/problems/longest-repeating-substring/) — rolling hash or suffix structures (Premium).
4. [718. Maximum Length of Repeated Subarray](https://leetcode.com/problems/maximum-length-of-repeated-subarray/) — binary search + hashing is one possible approach; DP is another.

Suggested order: 28 → 718 → 1062 → 1044.

For each problem, explain the hash convention, update formula, collision strategy, and complexity.

## 14. Reusable templates

### Polynomial hash
```python
def string_hash(s, base=31, mod=1_000_000_007):
    h = 0
    for ch in s:
        h = (h * base + ord(ch)) % mod
    return h
```

### Rolling update
```python
current = (current - outgoing * highest_power) % mod
current = (current * base + incoming) % mod
```

### Safe match check
```python
if window_hash == pattern_hash:
    if text[start:start + pattern_length] == pattern:
        answer.append(start)
```

### Final mental model
- Polynomial hash turns a string into a numeric signature.
- Rolling update removes the outgoing character and adds the incoming one.
- Prefix hash answers substring-hash queries.
- Hash matches can collide, so verify candidates.
- Binary search can combine with rolling hash when feasibility is monotonic.

**You have now completed all 15 Hashing theory concepts.**